# 🎯 YOLO26 객체 탐지로 포니봇 '물체 추적 주행' 만들기

**AI 기반 자율로봇 시스템 구현 과정 — 객체 탐지(Object Detection) 실습**

지난 전이학습 실습에서는 **분류(Classification)** 모델을 만들었습니다 — 화면 전체가 "무엇"인지만 알 수 있었죠.
이번에는 **객체 탐지(Detection)** 모델 YOLO26을 사용합니다 — "무엇이 **어디에, 얼마나 크게**" 있는지까지 알 수 있습니다.

| | 분류 (지난 실습) | 객체 탐지 (이번 실습) |
|---|---|---|
| 출력 | 클래스 1개 | 물체마다 (클래스, 위치 박스, 신뢰도) |
| 로봇 제어 | 카드 종류 → 명령 | 물체의 **위치** → 조향, **크기** → 전진/후진 |
| 결과 | 카드 보고 움직이는 로봇 | **물체를 따라다니는 로봇** 🐕 |

이 노트북에서 하는 일:
1. **YOLO26 체험** — 미리 학습된 모델(COCO 80종)로 이미지 속 물체 찾기
2. **출력 구조 이해** — end-to-end 모델의 `(1, 300, 6)` 출력 해부
3. **ONNX 내보내기** — 브라우저(ONNX Runtime Web)에서 돌아가는 형식으로 변환
4. **검증 & 다운로드** — 변환된 모델이 똑같이 동작하는지 확인 후 다운로드

```
[코랩: YOLO26 → ONNX]  →  [GitHub Pages 배포]  →  [웹앱: 실시간 탐지]  →  [위치/크기 → 명령]  →  [BLE]  →  [포니봇]
```

> 💡 **이번엔 학습이 없습니다!** COCO 데이터셋(33만 장)으로 미리 학습된 모델을 그대로 씁니다.
> 사람, 컵, 휴대폰, 공 등 80종을 이미 알고 있어요. (직접 학습은 도전과제에서!)

---
# 1단계. Ultralytics 설치 & YOLO26 불러오기

In [ ]:
!pip install -q ultralytics

from ultralytics import YOLO
import numpy as np

model = YOLO("yolo26n.pt")   # n = nano (가장 작고 빠른 버전, 첫 실행 시 자동 다운로드)
print("클래스 수:", len(model.names))
print("클래스 목록(일부):", dict(list(model.names.items())[:10]))

---
# 2단계. 이미지에서 물체 찾아보기

인터넷 이미지 한 장으로 YOLO26이 무엇을 찾아내는지 봅시다.

In [ ]:
# 샘플 이미지로 탐지 (직접 찍은 사진을 업로드해서 써도 됩니다!)
results = model.predict("https://ultralytics.com/images/bus.jpg", imgsz=640)

r = results[0]
print(f"찾은 물체 수: {len(r.boxes)}")
for box in r.boxes:
    cls  = model.names[int(box.cls)]
    conf = float(box.conf)
    x1, y1, x2, y2 = [int(v) for v in box.xyxy[0]]
    print(f"  {cls:12s} 신뢰도 {conf:.2f}  박스 ({x1},{y1})~({x2},{y2})")

# 결과 그려보기
from PIL import Image
Image.fromarray(r.plot()[:, :, ::-1])

In [ ]:
# (선택) 내 사진으로 테스트: 파일 업로드 후 탐지
from google.colab import files
up = files.upload()
for name in up:
    r = model.predict(name, imgsz=640)[0]
    display(Image.fromarray(r.plot()[:, :, ::-1]))

---
# 3단계. YOLO26이 특별한 이유 — 출력 구조 해부 🔬

옛날 YOLO는 수천 개의 후보 박스를 쏟아내서, **NMS(비최대 억제)**라는 후처리로
겹치는 박스를 걸러내야 했습니다. 브라우저에서 이걸 직접 구현하려면 코드가 꽤 복잡했죠.

**YOLO26은 end-to-end 모델**이라 NMS가 필요 없습니다.
모델이 곧바로 "정리된 최종 결과"를 줍니다:

```
출력 텐서: (1, 300, 6)
             │    │   └─ [x1, y1, x2, y2, 신뢰도, 클래스번호]
             │    └───── 최대 300개의 탐지 결과 (신뢰도 낮으면 0에 가까움)
             └────────── 배치 크기
```

→ 웹앱에서는 **신뢰도가 기준 이상인 행만 골라 쓰면 끝!** 후처리 코드가 몇 줄로 줄어듭니다.

---
# 4단계. ONNX로 내보내기

브라우저의 **ONNX Runtime Web**에서 실행할 수 있도록 ONNX 형식으로 변환합니다.
입력 크기는 브라우저 속도를 위해 **320×320**으로 줄입니다 (nano 모델 + 320이면 노트북 CPU로도 실시간에 가깝게 동작).

In [ ]:
IMG_SIZE = 320   # 브라우저 속도를 위해 작게 (기본 640)

path = model.export(format="onnx", imgsz=IMG_SIZE)   # end-to-end(NMS-free)가 기본값
print("내보내기 완료:", path)

import os
print(f"파일 크기: {os.path.getsize(path)/1024/1024:.1f} MB")

---
# 5단계. 변환 검증 — ONNX 모델이 똑같이 동작하나?

배포 전에 반드시! ONNX Runtime으로 직접 실행해서
**출력 형태가 정말 (1, 300, 6)인지**, 탐지가 정상인지 확인합니다.
(웹앱의 자바스크립트 코드가 이 형식을 가정하기 때문에 꼭 확인해야 합니다)

In [ ]:
import onnxruntime as ort_rt
import cv2

sess = ort_rt.InferenceSession("yolo26n.onnx")
inp_name  = sess.get_inputs()[0].name
out_name  = sess.get_outputs()[0].name
print("입력 이름:", inp_name, "| 형태:", sess.get_inputs()[0].shape)
print("출력 이름:", out_name, "| 형태:", sess.get_outputs()[0].shape)

# 전처리: 웹앱과 똑같이 (리사이즈 → RGB → 0~1 → CHW)
img = cv2.imread("bus.jpg") if os.path.exists("bus.jpg") else None
if img is None:
    import urllib.request
    urllib.request.urlretrieve("https://ultralytics.com/images/bus.jpg", "bus.jpg")
    img = cv2.imread("bus.jpg")
x = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
x = cv2.cvtColor(x, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
x = x.transpose(2, 0, 1)[None]           # (1, 3, 320, 320)

out = sess.run([out_name], {inp_name: x})[0]
print("\n실제 출력 형태:", out.shape)     # (1, 300, 6) 이어야 함!

# 신뢰도 0.5 이상인 탐지만 출력
det = out[0]
hits = det[det[:, 4] > 0.5]
print(f"신뢰도 0.5+ 탐지: {len(hits)}개")
for d in hits:
    x1, y1, x2, y2, conf, cls = d
    print(f"  {model.names[int(cls)]:10s} {conf:.2f}  ({x1:.0f},{y1:.0f})~({x2:.0f},{y2:.0f})")

assert out.shape[2] == 6, "출력 형식이 예상과 다릅니다!"
print("\n✅ 검증 통과 — 웹앱에서 그대로 사용 가능합니다")

> 📐 **좌표 해석**: 박스 좌표는 **320×320 입력 기준 픽셀값**입니다.
> 웹앱에서는 이 값을 320으로 나눠 0~1 비율로 만든 뒤, 화면 크기에 곱해 그립니다.

---
# 6단계. 다운로드

In [ ]:
from google.colab import files
files.download("yolo26n.onnx")
print("yolo26n.onnx 다운로드! (안 되면 왼쪽 📁 파일 탭에서 직접)")

---
# 🎉 완료! 다음 단계

1. `yolo26n.onnx`를 GitHub 저장소에 업로드 (웹앱 `yolo_tracking_drive.html`과 같은 위치)
2. GitHub Pages 주소로 접속 → 모델 로딩 → 블루투스 연결 → 카메라 시작
3. 추적할 물체(기본: person)를 카메라 앞에서 움직여 보세요 — 포니봇이 따라옵니다!

자세한 내용은 **`lab_yolo26_tracking.md`** 참고.

---
## 🥇 도전과제 미리보기: 나만의 물체 학습시키기

COCO에 없는 물체(예: 우리 반 마스코트 인형)를 추적하고 싶다면?
YOLO26도 전이학습이 됩니다:

```python
model = YOLO("yolo26n.pt")
model.train(data="my_dataset.yaml", epochs=50, imgsz=320)
model.export(format="onnx", imgsz=320)
```

라벨링(박스 그리기)이 필요해서 분류보다 데이터 준비가 오래 걸립니다.
Roboflow 같은 무료 라벨링 도구를 쓰면 편합니다. 웹앱은 클래스 이름 배열만 바꾸면 됩니다.